In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
print(f"Current working directory: {os.getcwd()}")
os.chdir('/content/drive/MyDrive/Colab Notebooks/LaViC')
print(f"New working directory: {os.getcwd()}")

Current working directory: /content
New working directory: /content/drive/MyDrive/Colab Notebooks/LaViC


In [ ]:
print("Items in current directory:")
for item in os.listdir('.'):
    print(item)

import tensorflow as tf

# Check for GPU availability
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print(f"GPU is available: {gpus}")
else:
    print("No GPU devices found.")

Items in current directory:
requirements.txt
data
src
out_distilled_amazon_home_test
lightning_logs
out_finetuned_amazon_home
GPU is available: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [ ]:
import os

# Path to requirements.txt
requirements_file = 'requirements.txt'

if os.path.exists(requirements_file):
    print(f"Found {requirements_file}. Checking modules...")
    with open(requirements_file, 'r') as f:
        for line in f:
            package_name = line.strip()
            if package_name and not package_name.startswith('#'):
                try:
                    # Attempt to import the module
                    __import__(package_name.split('==')[0].split('<')[0].split('>')[0].split('~')[0].split('!')[0].split('[')[0])
                    print(f"'{package_name}' is installed and importable.")
                except ImportError:
                    print(f"'{package_name}' is NOT installed or importable. You might need to install it.")
                except Exception as e:
                    print(f"Could not check '{package_name}' due to an error: {e}")
else:
    print(f"Error: {requirements_file} not found in the current directory ({os.getcwd()}).")

!pip install pytorch-lightning Pillow

os.environ['PYTORCH_ALLOC_CONF'] = 'expandable_segments:True'
print("PYTORCH_ALLOC_CONF set to expandable_segments:True")


Found requirements.txt. Checking modules...
'torch' is installed and importable.
'torchvision' is installed and importable.
'pytorch-lightning' is NOT installed or importable. You might need to install it.
'transformers' is installed and importable.
'peft' is installed and importable.
'Pillow' is NOT installed or importable. You might need to install it.
'tqdm' is installed and importable.
'sentencepiece' is installed and importable.
'accelerate' is installed and importable.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 857.3/857.3 kB 15.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 51.5 MB/s eta 0:00:00
PYTORCH_ALLOC_CONF set to expandable_segments:True


In [ ]:
!python src/knowledge_distillation.py \
  --model_name llava-hf/llava-v1.6-mistral-7b-hf \
  --train_data ./data/item2meta_train_amazon_home.json \
  --val_data ./data/item2meta_valid.jsonl \
  --train_images_dir ./data/amazon_home_train_images_subset \
  --val_images_dir ./data/valid_images \
  --output_dir ./out_distilled_amazon_home_test \
  --lr 5e-5 \
  --weight_decay 1e-5 \
  --num_epochs 2 \
  --batch_size 4

Streaming output truncated to the last 5000 lines.
                                                               1.802            
                                                               train_loss_epoch:
Epoch 1/1  ━━━━━━━━━━━━━━╺━ 3933/4476 0:55:25 •       1.71it/s v_num: 0.000     
                                      0:05:19                  train_loss_step: 
                                                               0.704 val_loss:  
                                                               0.589            
                                                               val_perplexity:  
                                                               1.802            
                                                               train_loss_epoch:
Epoch 1/1  ━━━━━━━━━━━━━━╺━ 3934/4476 0:55:26 •       1.71it/s v_num: 0.000     
                                      0:05:18                  train_loss_step: 
                                                          

In [ ]:
!python src/prompt_tuning.py \
  --base_model_name llava-hf/llava-v1.6-mistral-7b-hf \
  --model_dir ./out_distilled_amazon_home_test/vision_lora_adapter_best \
  --candidate_type candidates_st \
  --finetune_output_dir ./out_finetuned_amazon_home \
  --max_length 2048 \
  --batch_size 1 \
  --lr 5e-5 \
  --weight_decay 1e-5 \
  --num_epochs 1 \
  --item_meta_path ./data/item2meta_train_amazon_home.json \
  --image_dir ./data/amazon_home_train_images_subset \
  --category amazon_home

Streaming output truncated to the last 5000 lines.
Epoch 0/0  ━━━━━━╸━━━━━━━━━ 1334/3077 0:17:14 •       1.52it/s v_num: 2.000     
                                      0:19:08                  train_loss_step: 
Epoch 0/0  ━━━━━━╸━━━━━━━━━ 1335/3077 0:17:15 •       1.52it/s v_num: 2.000     
                                      0:19:07                  train_loss_step: 
Epoch 0/0  ━━━━━━╸━━━━━━━━━ 1336/3077 0:17:16 •       1.52it/s v_num: 2.000     
                                      0:19:06                  train_loss_step: 
Epoch 0/0  ━━━━━━╸━━━━━━━━━ 1337/3077 0:17:16 •       1.52it/s v_num: 2.000     
                                      0:19:06                  train_loss_step: 
Epoch 0/0  ━━━━━━╸━━━━━━━━━ 1338/3077 0:17:17 •       1.52it/s v_num: 2.000     
                                      0:19:07                  train_loss_step: 
Epoch 0/0  ━━━━━━╸━━━━━━━━━ 1339/3077 0:17:18 •       1.51it/s v_num: 2.000     
                                      0:19:10             

In [ ]:
!python ./src/baseline_llava_zero_shot.py \
  --base_model_name llava-hf/llava-v1.6-mistral-7b-hf \
  --candidate_type candidates_st \
  --item_meta_path ./data/item2meta_train_amazon_home.json \
  --category amazon_home \
  --output_dir ./out_baseline_llava_title_only_home

[INFO] Loading base LLaVA model...
config.json: 1.25kB [00:00, 3.98MB/s]
You are using a model of type llava_next to instantiate a model of type llava. This is not supported for all configurations of models and can yield errors.
model.safetensors.index.json: 70.2kB [00:00, 123MB/s]
Fetching 4 files: 100% 4/4 [00:45<00:00, 11.46s/it]
Download complete: 100% 15.1G/15.1G [00:45<00:00, 329MB/s] 
Loading weights: 100% 686/686 [00:01<00:00, 625.26it/s, Materializing param=model.vision_tower.vision_model.pre_layrnorm.weight]
LlavaForConditionalGeneration LOAD REPORT from: llava-hf/llava-v1.6-mistral-7b-hf
Key           | Status     |  | 
--------------+------------+--+-
image_newline | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
generation_config.json: 100% 116/116 [00:00<00:00, 663kB/s]
processor_config.json: 100% 176/176 [00:00<00:00, 1.02MB/s]
chat_template.json: 100% 694/694 [00:00<00:00, 3.97MB/